In [17]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from src.embeddings.embed_chunks import MODEL_CONFIGS

EMBEDDING_CONFIG = MODEL_CONFIGS["bgebase"]
QUERY_PREFIX = EMBEDDING_CONFIG["query_prefix"]

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
EVAL_DIR = PROJECT_ROOT / "data/evaluation"

In [25]:
bge = pd.read_json(Path(EVAL_DIR / "dense_retrieval" / "bgebase" / "v1" / "evaluation.jsonl"), lines=True)

In [27]:
bm25 = pd.read_json(Path(EVAL_DIR / "bm25_retrieval" / "v1" / "evaluation.jsonl"), lines=True)

In [28]:
bge.head()

,company,ticker,question_type,question,expected_chunk_ids,expected_chunk_count,expected_ranks,recall@1,hit@1,complete@1,...,latency_seconds,retrieved_chunk_ids,run_number,timestamp,embedding_model,reranker_model,retrieval_method,retrieval_parameters,evaluation_dataset,device
0,"Aurora Innovation, Inc.",AUR,single_narrative,What is Aurora's mission for delivering self-d...,[AUR-2025-CHUNK-000002],1,{'AUR-2025-CHUNK-000002': 1},1.0,1,1,...,0.044940,"[AUR-2025-CHUNK-000002, AUR-2025-CHUNK-000037,...",1,2026-08-19 00:14:08.071514+02:00,BAAI/bge-base-en-v1.5,NaN,dense_retrieve,"{'top_k': 30, 'query_prefix': 'Represent this ...",data/evaluation/test_queries.jsonl,cpu
1,"Aurora Innovation, Inc.",AUR,single_narrative,Who founded Aurora in 2017?,[AUR-2025-CHUNK-000002],1,{'AUR-2025-CHUNK-000002': 2},0.0,0,0,...,0.105998,"[AUR-2025-CHUNK-000014, AUR-2025-CHUNK-000002,...",1,2026-08-19 00:14:08.071514+02:00,BAAI/bge-base-en-v1.5,NaN,dense_retrieve,"{'top_k': 30, 'query_prefix': 'Represent this ...",data/evaluation/test_queries.jsonl,cpu
2,"Aurora Innovation, Inc.",AUR,single_narrative,What vehicle types is the Aurora Driver design...,[AUR-2025-CHUNK-000002],1,{'AUR-2025-CHUNK-000002': 4},0.0,0,0,...,0.086014,"[AUR-2025-CHUNK-000016, AUR-2025-CHUNK-000021,...",1,2026-08-19 00:14:08.071514+02:00,BAAI/bge-base-en-v1.5,NaN,dense_retrieve,"{'top_k': 30, 'query_prefix': 'Represent this ...",data/evaluation/test_queries.jsonl,cpu
3,"Aurora Innovation, Inc.",AUR,single_narrative,Which three target markets does Aurora identif...,[AUR-2025-CHUNK-000002],1,{'AUR-2025-CHUNK-000002': 4},0.0,0,0,...,0.088831,"[AUR-2025-CHUNK-000017, AUR-2025-CHUNK-000012,...",1,2026-08-19 00:14:08.071514+02:00,BAAI/bge-base-en-v1.5,NaN,dense_retrieve,"{'top_k': 30, 'query_prefix': 'Represent this ...",data/evaluation/test_queries.jsonl,cpu
4,"Aurora Innovation, Inc.",AUR,single_narrative,Why did Aurora choose trucking as the first co...,[AUR-2025-CHUNK-000003],1,{'AUR-2025-CHUNK-000003': 2},0.0,0,0,...,0.103971,"[AUR-2025-CHUNK-000008, AUR-2025-CHUNK-000003,...",1,2026-08-19 00:14:08.071514+02:00,BAAI/bge-base-en-v1.5,NaN,dense_retrieve,"{'top_k': 30, 'query_prefix': 'Represent this ...",data/evaluation/test_queries.jsonl,cpu


In [34]:
TOP_K = 100
join_columns = ["ticker", "question"]

comparison = bge[
    join_columns + ["expected_chunk_ids", "retrieved_chunk_ids"]
].merge(
    bm25[join_columns + ["retrieved_chunk_ids"]],
    on=join_columns,
    how="inner",
    suffixes=("_bge", "_bm25"),
    validate="one_to_one",
)


def oracle_metrics(row):
    expected = set(row["expected_chunk_ids"])

    bge = set(row["retrieved_chunk_ids_bge"][:TOP_K])
    bm25 = set(row["retrieved_chunk_ids_bm25"][:TOP_K])

    oracle = bge | bm25
    relevant = expected & oracle

    return pd.Series({
        "oracle_recall": len(relevant) / len(expected),
        "oracle_hit": int(len(relevant) > 0),
        "oracle_complete": int(len(relevant) == len(expected)),
    })


def compare_expected(row):
    expected = set(row["expected_chunk_ids"])
    bge = set(row["retrieved_chunk_ids_bge"][:TOP_K])
    bm25 = set(row["retrieved_chunk_ids_bm25"][:TOP_K])

    bge_hits = expected & bge
    bm25_hits = expected & bm25

    return pd.Series({
        "both": sorted(bge_hits & bm25_hits),
        "bge_only": sorted(bge_hits - bm25_hits),
        "bm25_only": sorted(bm25_hits - bge_hits),
        "neither": sorted(expected - (bge_hits | bm25_hits)),
    })


comparison = comparison.join(
    comparison.apply(compare_expected, axis=1)
)

for column in ["both", "bge_only", "bm25_only", "neither"]:
    comparison[f"{column}_count"] = comparison[column].map(len)


summary = pd.Series({
    "Expected chunks": comparison["expected_chunk_ids"].map(len).sum(),
    "Found by both": comparison["both_count"].sum(),
    "BGE only": comparison["bge_only_count"].sum(),
    "BM25 only": comparison["bm25_only_count"].sum(),
    "Missed by both": comparison["neither_count"].sum(),
})

print(f"\nExpected-chunk coverage @ {TOP_K}")
print(summary.to_string())


question_summary = pd.Series({
    "Questions where BM25 rescues evidence":
        (comparison["bm25_only_count"] > 0).sum(),

    "Questions where BGE rescues evidence":
        (comparison["bge_only_count"] > 0).sum(),

    "Questions with evidence found by both":
        (comparison["both_count"] > 0).sum(),

    "Questions with evidence missed by both":
        (comparison["neither_count"] > 0).sum(),
})

print("\nQuestion-level comparison")
print(question_summary.to_string())


comparison[
    [
        "ticker",
        "question",
        "expected_chunk_ids",
        "both",
        "bge_only",
        "bm25_only",
        "neither",
    ]
]


Expected-chunk coverage @ 100
Expected chunks    430
Found by both      284
BGE only            51
BM25 only           42
Missed by both      53

Question-level comparison
Questions where BM25 rescues evidence      38
Questions where BGE rescues evidence       44
Questions with evidence found by both     231
Questions with evidence missed by both     47


,ticker,question,expected_chunk_ids,both,bge_only,bm25_only,neither
0,AUR,What is Aurora's mission for delivering self-d...,[AUR-2025-CHUNK-000002],[AUR-2025-CHUNK-000002],[],[],[]
1,AUR,Who founded Aurora in 2017?,[AUR-2025-CHUNK-000002],[AUR-2025-CHUNK-000002],[],[],[]
2,AUR,What vehicle types is the Aurora Driver design...,[AUR-2025-CHUNK-000002],[AUR-2025-CHUNK-000002],[],[],[]
3,AUR,Which three target markets does Aurora identif...,[AUR-2025-CHUNK-000002],[AUR-2025-CHUNK-000002],[],[],[]
4,AUR,Why did Aurora choose trucking as the first co...,[AUR-2025-CHUNK-000003],[AUR-2025-CHUNK-000003],[],[],[]
...,...,...,...,...,...,...,...
295,OUST,How do Ouster's four target markets relate to ...,"[OUST-2025-CHUNK-000001, OUST-2025-CHUNK-000002]","[OUST-2025-CHUNK-000001, OUST-2025-CHUNK-000002]",[],[],[]
296,OUST,What does Ouster's filing show about both its ...,"[OUST-2025-CHUNK-000368, OUST-2025-CHUNK-000001]",[],"[OUST-2025-CHUNK-000001, OUST-2025-CHUNK-000368]",[],[]
297,OUST,"How do Ouster's unified Physical AI platform, ...","[OUST-2025-CHUNK-000003, OUST-2025-CHUNK-00000...","[OUST-2025-CHUNK-000001, OUST-2025-CHUNK-000002]",[],[OUST-2025-CHUNK-000003],[]
298,OUST,"How do Ouster's target markets, manufacturing ...","[OUST-2025-CHUNK-000003, OUST-2025-CHUNK-00000...","[OUST-2025-CHUNK-000001, OUST-2025-CHUNK-000002]",[],[],[OUST-2025-CHUNK-000003]


In [35]:

oracle = comparison.apply(oracle_metrics, axis=1)

print(f"Oracle union @{TOP_K}")
print(f"Recall:   {oracle['oracle_recall'].mean():.4f}")
print(f"Hit:      {oracle['oracle_hit'].mean():.4f}")
print(f"Complete: {oracle['oracle_complete'].mean():.4f}")

TypeError: 'set' object is not subscriptable